# Introduction to LakeFlow Jobs

**LakeFlow Jobs** is the Databricks workflow orchestration engine — the layer that schedules, coordinates, and monitors the execution of data engineering work.

It is part of the broader **Lakeflow** product family alongside Lakeflow Declarative Pipelines (formerly Delta Live Tables). While Declarative Pipelines handle continuous or triggered ingestion and transformation, **Jobs** is the tool you reach for when you need to orchestrate multi-step workflows, run notebooks on a schedule, or chain together heterogeneous task types with dependencies.

> In older Databricks documentation you will see this called **Databricks Workflows** or simply **Jobs**. LakeFlow Jobs is the current branding.

## Why LakeFlow Jobs?

Running a notebook manually works fine for exploration. In production, you need:

- **Scheduling** — run at midnight, every hour, or when a file lands in a Volume.
- **Reliability** — automatic retries on failure, alerting, and repair runs.
- **Dependency management** — Task B only runs after Task A succeeds.
- **Isolation** — each run gets its own ephemeral job cluster, not an all-purpose cluster that other users share.
- **Observability** — full run history, logs, and duration metrics in one place.

## Anatomy of a Job

```
Job
├── Trigger          (schedule / file arrival / manual / API)
├── Task 1           (e.g. Notebook — ingest raw data)
│   └── depends on: (none)
├── Task 2           (e.g. Notebook — clean and transform)
│   └── depends on: Task 1
└── Task 3           (e.g. SQL — refresh a dashboard table)
    └── depends on: Task 2
```

### Key concepts

| Concept | Description |
|---|---|
| **Job** | The top-level container. Has a name, trigger, and one or more tasks. |
| **Task** | A single unit of work — a notebook, a SQL statement, a Python script, a pipeline run, etc. |
| **Run** | One execution of a job. Every triggered or manual invocation creates a new run. |
| **Task run** | The execution of a single task within a run. |
| **Repair run** | Re-execute only the failed (and downstream) tasks of a previous run without restarting from scratch. |

## Task Types

LakeFlow Jobs supports a wide variety of task types, making it the central orchestrator for the entire lakehouse:

| Task type | Use case |
|---|---|
| **Notebook** | Run a `.ipynb` or Databricks notebook — the most common task type. |
| **Python script** | Execute a `.py` file directly on the cluster. |
| **SQL** | Run a SQL statement or query against a SQL warehouse. |
| **Delta Live Tables pipeline** | Trigger a Lakeflow Declarative Pipelines run as a task. |
| **dbt** | Run a dbt project as a task (requires dbt Cloud or dbt Core setup). |
| **Spark JAR** | Submit a compiled Spark application. |
| **Run Job** | Trigger another job as a sub-task — enables modular, reusable job composition. |
| **Condition** | Evaluate an expression to branch the DAG — e.g., run Task C only if a flag is set. |

## Triggers

How a job starts determines its trigger type:

| Trigger | Description |
|---|---|
| **Scheduled (Cron)** | Run on a fixed cadence — hourly, daily, every Monday at 06:00, etc. |
| **File arrival** | Start whenever a new file appears in a Unity Catalog Volume or cloud storage path. |
| **Continuous** | Always-on — restart immediately after each run completes. |
| **Manual / API** | Triggered by a human click in the UI or a REST API call from an external system. |

## Compute Options

Each task in a job specifies its own compute. You have three options:

| Option | Description |
|---|---|
| **Job cluster (ephemeral)** | A cluster created at run time and terminated when the task finishes. Fresh, isolated, cost-efficient — the recommended approach for production. |
| **Serverless compute** | No cluster to configure. Databricks manages scaling automatically. Fastest startup, pay-per-use. |
| **All-purpose cluster** | Attach to an existing interactive cluster. Useful for development but not recommended for production — shared state and higher cost. |

> **Best practice:** Use **serverless compute** or a **job cluster** for production jobs. Reserve all-purpose clusters for interactive development.

## Task Dependencies — the DAG

Tasks in a job form a **Directed Acyclic Graph (DAG)**.  
Each task declares which other tasks it depends on. Databricks resolves the execution order automatically, running independent tasks in parallel where possible.

```
       [Ingest]          ← no dependency, runs first
          │
    ┌─────┴─────┐
[Transform A] [Transform B]   ← both depend on Ingest, run in parallel
    └─────┬─────┘
          │
       [Report]          ← depends on both transforms, runs last
```

You can also use **conditional tasks** (`if/else` branching) and **for-each tasks** (loop over a parameter list) to build dynamic workflows.

## Job Parameters and Task Values

Jobs support two mechanisms for passing data between tasks:

**Job parameters** — key-value pairs set when the job is triggered. Every task in the run can read them.

```python
# Inside a notebook task — read a job parameter
env = dbutils.widgets.get("environment")   # e.g. "prod" or "dev"
```

**Task values** — a task can publish a value, and a downstream task can consume it. Useful for passing IDs, row counts, or file paths between steps.

```python
# Task A — publish a value
dbutils.jobs.taskValues.set(key="row_count", value=df.count())

# Task B (downstream) — consume it
count = dbutils.jobs.taskValues.get(taskKey="TaskA", key="row_count", default=0)
```

## What's coming in the next notebooks

This series will progressively build up practical LakeFlow Jobs knowledge:

| Notebook | Topic |
|---|---|
| **02** | Creating and configuring jobs — UI walkthrough and JSON job definition |
| **03** | Multi-task workflows — building a DAG with dependencies |
| **04** | Job parameters and task values — dynamic, data-driven pipelines |
| **05** | Triggers — schedules, file arrival, and continuous mode |
| **06** | Error handling — retries, alerts, repair runs, and conditional branching |
| **07** | Monitoring and observability — run history, logs, and metrics |